# Analyse exploratoire des factures
Objectif : comprendre la structure des données et identifier, sans utiliser
les étiquettes, les signaux qui distinguent une facture anormale.

In [118]:
import pandas as pd

df_contrats = pd.read_csv("../data/contrats.csv")
df_factures = pd.read_csv("../data/factures.csv",parse_dates=["date_facture"])
df_factures.dtypes

id_facture                        str
identifiant_contrat               str
date_facture           datetime64[us]
consommation_kwh                int64
montant                       float64
index_debut                     int64
index_fin                       int64
anomalie                          str
dtype: object

On exclut ensuite les anomalies: pop retire la colonne du DataFrame et la range dans une variable à part 

In [119]:
etiquettes = df_factures.pop("anomalie")

In [120]:
df_factures.info()

<class 'pandas.DataFrame'>
RangeIndex: 572 entries, 0 to 571
Data columns (total 7 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   id_facture           572 non-null    str           
 1   identifiant_contrat  572 non-null    str           
 2   date_facture         572 non-null    datetime64[us]
 3   consommation_kwh     572 non-null    int64         
 4   montant              572 non-null    float64       
 5   index_debut          572 non-null    int64         
 6   index_fin            572 non-null    int64         
dtypes: datetime64[us](1), float64(1), int64(3), str(2)
memory usage: 37.6 KB


In [121]:
df_factures.describe()

,date_facture,consommation_kwh,montant,index_debut,index_fin
count,572,572.000000,572.000000,572.000000,572.000000
mean,2025-06-14 15:26:26.013986,312.639860,107.720035,30969.680070,31275.592657
min,2025-01-01 00:00:00,65.000000,24.870000,12473.000000,12845.000000
25%,2025-03-01 00:00:00,134.000000,45.507500,22314.750000,22500.250000
50%,2025-06-01 00:00:00,229.000000,72.050000,30802.000000,31041.000000
75%,2025-09-01 00:00:00,370.250000,110.732500,39848.250000,40144.500000
max,2025-12-01 00:00:00,6420.000000,2703.240000,53560.000000,54064.000000
std,NaN,442.522767,195.485656,10414.432724,10438.690412


La consommation médiane est de 229 kWh et 75 % des factures sont sous 370 kWh, mais le maximum atteint 6420 kWh, soit environ 17 fois le 3e quartile. Pour le montant on arrive même à 27x le montant du 3è quartile
L'index début et index n'apportent pas de mesure claire en l'état via ce tableau, à analyser plus tard

## Contrôles de qualité des données

In [ ]:
print("Liste des valeurs manquantes sur contrats :\n",df_contrats.isna().sum())
print("\nListe des valeurs manquantes sur factures :\n",df_factures.isna().sum())
print("\nFactures - clé unique :", df_factures["id_facture"].is_unique)
print("\nContrats - clé unique :", df_contrats["identifiant"].is_unique)
print("\nIntégrité référentielle :", df_factures["identifiant_contrat"].isin(df_contrats["identifiant"]).all())
print("\nCohérence actif / résiliation :", (df_contrats["actif"] == df_contrats["mois_resiliation"].isna()).all())
print("\nPart de contrats actifs :", df_contrats["actif"].mean())

Liste des valeurs manquantes sur contrats :
 identifiant          0
nom_client           0
puissance_kva        0
prix_kwh             0
actif                0
mois_resiliation    47
dtype: int64

Liste des valeurs manquantes sur factures :
 id_facture             0
identifiant_contrat    0
date_facture           0
consommation_kwh       0
montant                0
index_debut            0
index_fin              0
dtype: int64

Factures - clé unique : True

Contrats - clé unique : True

Intégrité référentielle : True


## Bilan 
- Valeurs manquantes : uniquement mois_resiliation, pour les contrats actifs. C’est attendu, pas un défaut.
- Clés primaires uniques : contrats OK, factures OK
- Intégrité référentielle factures → contrats OK
- Cohérence actif / mois de résiliation OK (90% actifs)

→ Les Données sont donc exploitables.